In [1]:
import os
import fastf1
import pandas as pd
from pathlib import Path

In [2]:
# Cache beállítása és engedélyezése
os.makedirs('cache', exist_ok=True)
fastf1.Cache.enable_cache('cache')

In [3]:
# car_data mappa létrehozása a jelenlegi könyvtárban
output_dir = Path("car_data")
output_dir.mkdir(exist_ok=True)

In [4]:
# Szezon beállítása
YEAR = 2018
YY = str(YEAR)[-2:]

In [5]:
# Lekérjük a versenynaptárat a futamok pontos száma miatt (biztonságosabb, mint a fix 22)
schedule = fastf1.get_event_schedule(YEAR)
total_races = schedule[schedule['EventFormat'] != 'testing'].shape[0]

for i in range(1, total_races + 1):
    print(f"[{i}/{total_races}] {YEAR}. szezon {i}. futamának betöltése...")
    try:
        race = fastf1.get_session(YEAR, i, 'R')
        # Elég csak a telemetriát betölteni (laps és weather nélkül gyorsabb)
        race.load(telemetry=True, laps=False, weather=False)
    except Exception as e:
        print(f"Hiba a(z) {i}. futam betöltésekor: {e}")
        continue

    # 3. Adott futam metaadatainak kinyerése
    event_name = race.event['EventName']
    location = race.event['Location']
    round_number = race.event['RoundNumber']

    all_car_data = []

    # 4. Minden versenyző car_data-jának lekérése iterálással
    # A race.car_data egy dictionary, ahol a kulcs a versenyző rajtszáma (pl. '1', '16')
    for driver_num, data in race.car_data.items():
        if data.empty:
            continue

        # Készítünk egy másolatot a DataFrame-ről, hogy ne az eredetit módosítsuk
        df = data.copy()
        
        # Opcionális: megpróbáljuk a 3 betűs rövidítést (pl. 'VER') hozzáadni, 
        # ha nem sikerül, marad a rajtszám.
        try:
            driver_info = race.get_driver(driver_num)
            df['Driver'] = driver_info['Abbreviation']
        except Exception:
            df['Driver'] = driver_num
            
        # Hozzáadjuk a kért új oszlopokat
        df['EventName'] = event_name
        df['Location'] = location
        df['Round'] = round_number
        
        all_car_data.append(df)

    if all_car_data:
        # Összefűzzük az összes pilóta adatát egy nagy DataFrame-be
        car_data_all = pd.concat(all_car_data, ignore_index=True)
        
        # 5. Fájl mentése a kért formátumban: pl. car_data_22_10.parquet
        file_path = output_dir / f"car_data_{YY}_{i}.parquet"
        car_data_all.to_parquet(file_path)
        print(f"--> Sikeresen mentve: {file_path}")

        # Memória felszabadítása a következő futam előtt
        del car_data_all, all_car_data

print("Minden elérhető futam feldolgozása befejeződött!")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for car_data
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...


[1/21] 2018. szezon 1. futamának betöltése...


core        WARNING 	Car position data is unavailable!
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...


--> Sikeresen mentve: car_data\car_data_18_1.parquet
[2/21] 2018. szezon 2. futamának betöltése...


core        WARNING 	Car telemetry data is unavailable!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
core        WARNING 	Car position data is unavailable!
core        WARNING 	Failed to determine `Session.t0_date`!
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['5', '77', '44', '10', '20', '27', '14', '2', '9', '31', '55', '16', '8', '18', '35', '11', '28', '7', '33', '3']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data


...

--> Sikeresen mentve: car_data\car_data_18_19.parquet
[20/21] 2018. szezon 20. futamának betöltése...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '33', '7', '3', '77', '5', '16', '8', '20', '11', '28', '55', '10', '2', '31', '35',

--> Sikeresen mentve: car_data\car_data_18_20.parquet
[21/21] 2018. szezon 21. futamának betöltése...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '5', '33', '3', '77', '55', '16', '11', '8', '20', '14', '28', '18', '2', '35', '10'

--> Sikeresen mentve: car_data\car_data_18_21.parquet
Minden elérhető futam feldolgozása befejeződött!
